In [29]:
! pip install langchain langchain-groq langchain-community langchain-core --quiet

In [30]:
import os
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser


def generate_social_media_post_langchain(platform, topic, max_tokens=300, add_emoji=True):
    if "GROQ_API_KEY" not in os.environ or not os.environ["GROQ_API_KEY"]:
        return "Error: GROQ API Key is not set in environment variables. Please provide it in the Gradio UI."

    emoji_instruction = "Include relevant emojis in the post." if add_emoji else "Do not include any emojis."

    if platform.lower() == 'linkedin':
        template_content = f"You are an expert social media manager specializing in LinkedIn. Your task is to craft professional and engaging posts related to the provided topic. {emoji_instruction}\n\nCreate a professional LinkedIn post about: {{topic}}. Ensure it is complete, concise, and thought-provoking, suitable for a professional audience."
    elif platform.lower() == 'instagram':
        template_content = f"You are an expert social media manager specializing in Instagram. Your task is to craft visually appealing and engaging captions for the provided topic. {emoji_instruction}. Also, suggest 3-5 relevant hashtags.\n\nCreate an engaging Instagram post caption about: {{topic}}. Make it complete, catchy and encouraging interaction."
    elif platform.lower() == 'facebook':
        template_content = f"You are an expert social media manager specializing in Facebook. Your task is to craft friendly and informative posts related to the provided topic. {emoji_instruction}\n\nCreate a friendly and informative Facebook post about: {{topic}}. Ensure it is complete and encourages discussion in the comments."
    else:
        return "Invalid platform. Please choose 'LinkedIn', 'Instagram', or 'Facebook'."

    llm_social = ChatGroq(model="llama-3.1-8b-instant", temperature=0.7, max_tokens=max_tokens)
    prompt_social = PromptTemplate(
        input_variables=["topic"],
        template=template_content
    )
    parser_social = StrOutputParser()

    chain_social = prompt_social | llm_social | parser_social

    response = chain_social.invoke({"topic": topic})
    return response

print("Welcome to the Social Media Post Generator!")

Welcome to the Social Media Post Generator!


In [31]:
import gradio as gr

In [32]:
def social_media_post_generator_gradio(groq_api_key_from_ui, platform, topic, max_tokens, add_emoji):
    if groq_api_key_from_ui and isinstance(groq_api_key_from_ui, str):
        os.environ["GROQ_API_KEY"] = groq_api_key_from_ui
    elif "GROQ_API_KEY" in os.environ:
        del os.environ["GROQ_API_KEY"]
    generated_post = generate_social_media_post_langchain(
        platform=platform,
        topic=topic,
        max_tokens=max_tokens,
        add_emoji=add_emoji
    )
    return generated_post

In [34]:
iface = gr.Interface(
    fn=social_media_post_generator_gradio,
    inputs=[
        gr.Textbox(label="GROQ API Key", type="password", placeholder="Enter your Groq API Key here..."), # Added API key input
        gr.Radio(['linkedin', 'instagram', 'facebook'], label="Platform"),
        gr.Textbox(label="Topic", placeholder="Enter the topic for your post..."),
        gr.Slider(minimum=50, maximum=500, step=10, value=300, label="Maximum Token Length"),
        gr.Checkbox(label="Include Emojis", value=True)
    ],
    outputs=gr.Textbox(label="Generated Social Media Post", lines=15),
    title="Social Media Post Generator",
    description="Generate engaging social media posts using Groq and LangChain."
)

iface.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://8d5f44c653b973bd06.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7868 <> https://8d5f44c653b973bd06.gradio.live
